# Delinquency Prediction Dataset - EDA Project

This notebook performs basic EDA, missing value analysis, and cleaning steps.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path = 'Delinquency_prediction_dataset.xlsx'
df = pd.read_excel(file_path)
df.head()

## 2. Dataset Overview

In [ ]:
print('Shape:', df.shape)
print(df.info())
print(df.describe())

## 3. Missing Data Analysis

In [ ]:
missing = df.isnull().sum()
missing_percent = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_percent})
missing_table[missing_table['Missing Count'] > 0]

## 4. Data Cleaning / Imputation
Median imputation is used for numerical missing values because it is more robust when outliers are present.

In [ ]:
df_clean = df.copy()

for col in ['Income', 'Credit_Score', 'Loan_Balance']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Standardize Employment_Status labels
df_clean['Employment_Status'] = df_clean['Employment_Status'].replace({
    'EMP': 'Employed',
    'employed': 'Employed',
    'retired': 'Retired'
})

# Cap Credit_Utilization above 1.00
df_clean['Credit_Utilization'] = np.where(df_clean['Credit_Utilization'] > 1, 1, df_clean['Credit_Utilization'])

df_clean.isnull().sum()

## 5. Target Distribution

In [ ]:
df_clean['Delinquent_Account'].value_counts().plot(kind='bar', title='Delinquent Account Distribution')
plt.xlabel('Delinquent Account')
plt.ylabel('Customer Count')
plt.show()

## 6. Correlation with Target

In [ ]:
numeric_cols = df_clean.select_dtypes(include=['number']).columns
corr = df_clean[numeric_cols].corr()['Delinquent_Account'].sort_values(key=abs, ascending=False)
corr

## 7. Key Business Insights

- The dataset is imbalanced: delinquent customers are much fewer than non-delinquent customers.
- Missing values exist in Income, Credit_Score, and Loan_Balance.
- Employment_Status had inconsistent labels and should be standardized before modeling.
- Credit_Utilization has a few values above the expected 0-1 range, so they were capped.
- Correlations with the target are weak, so future modeling should test non-linear models and feature interactions.

## 8. Save Cleaned Dataset

In [ ]:
df_clean.to_excel('Cleaned_Delinquency_prediction_dataset.xlsx', index=False)